# Feature Engineering with scikit-learn Pipelines

Moving preprocessing into scikit-learn pipelines makes your workflow easier to reuse, validate, and deploy.

Key benefits:
1. **Modularity**: each transformation is its own step.
2. **Consistency**: the same steps run for training and new data.
3. **Safer evaluation**: fit/transform separation reduces leakage risk.
4. **Model integration**: preprocessing and model training can run in one workflow.
5. **Extensibility**: custom transformers plug into the same API.

This notebook focuses on turning manual pandas cleanup steps into reusable scikit-learn-compatible transformers and pipelines.


> **Checkpoint:**
> You can explain the difference between `fit`, `transform`, and `fit_transform`.

> **Common pitfalls:**
> - Fitting preprocessing on combined train+validation data.
> - Writing transformers that mutate inputs in place.
> - Losing feature-name traceability after transformation.

> **Self-check:**
> Could you reuse your pipeline on unseen data without changing code?


## Visual Guide: Pipeline Architecture

```mermaid
flowchart TD
    A["Raw Seattle weather data"]
    B["Custom transformers"]
    C["Data cleaning pipeline"]
    D["Numeric branch"]
    E["Categorical branch"]
    F["ColumnTransformer output"]
    G["Fit on train and transform new data"]

    A --> B --> C
    C --> D --> F
    C --> E --> F
    F --> G
```


### Loading data and preparation steps

We will use the Seattle Weather Dataset again in this notebook and use Pydantic to validate whether the data looks as expected.  

If you've done notebook 01, the first cells will look familiar. Let's import the data and write our Pydantic model.  


In [36]:
# pandas handles tabular data throughout the notebook.
import pandas as pd

# We validate the final schema against real Python datetime objects.
from datetime import datetime
from pydantic import BaseModel

# We split the dataset so we can practice the train/new-data workflow.
from sklearn.model_selection import train_test_split

# `re` lets us clean shorthand weather labels with regular expressions.
import re

# NumPy is used later for mathematical transformations like square root.
import numpy as np

In [37]:
# Import the data
seattle_weather_data = pd.read_csv("../data/seattle-weather_raw.csv")
seattle_weather_data.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012/01/01,0.0,12.8,41.0 F,4.710858,drizzle
1,2012/01/02,10.9,NaN,37.04 F,4.527453,r
2,2012/01/03,0.8,11.7,44.96 F,2.315752,r.
3,2012/01/03,0.8,11.7,44.96 F,2.315752,r.
4,2012/01/04,20.3,12.2,42.08 F,4.719543,r


Let's remember the expected data types for the Pydantic model:  

| Column | Expected data type |
|---|---|
| `date` | datetime |
| `precipitation` | float |
| `temp_max` | float |
| `temp_min` | float |
| `wind` | float |
| `weather` | string |  


In [38]:
class DataValidation(BaseModel):
    date: datetime
    precipitation: float
    temp_max: float
    temp_min: float
    wind: float
    weather: str

We define the **validation function** again so we can turn a pandas DataFrame into dictionaries one row at a time (see [notebook 01](../01-feature-engineering-with-pandas/01-feature-engineering-with-pandas.ipynb)):


In [39]:
def data_validation(df: pd.DataFrame, data_schema) -> pd.DataFrame:
    class DataFrameValidation(BaseModel):
        # Pydantic will validate every row against the schema we pass in.
        df_as_dict: list[data_schema]

    # Convert the DataFrame into a list of row dictionaries because
    # Pydantic validates normal Python objects, not pandas objects directly.
    df_as_dict = df.to_dict(orient="records")

    # If one row has the wrong type, this line will raise a validation error.
    DataFrameValidation(df_as_dict=df_as_dict)

    # Returning the original DataFrame keeps this helper easy to reuse in a pipeline-like workflow.
    return df

Now let's start with the feature engineering—this time with **scikit-learn**!


## Dealing with messy data

scikit-learn does not include a built-in transformer for every custom cleanup task. In those cases, we create a **custom transformer** that follows the estimator API.

Minimum requirements:
- inherit from `BaseEstimator` and `TransformerMixin`,
- implement `fit(self, X, y=None)` and return `self`,
- implement `transform(self, X, y=None)` and return transformed data.

Practical rules:
- copy your input (`X.copy()`) before editing values  
→ prevents modifying the original DataFrame outside the transformer.

- keep transformation logic deterministic  
→ ensures the same input always produces the same output, which is required for reliable pipelines and reproducibility.

- avoid hidden side effects  
→ prevents unexpected changes to external variables or data that could break pipeline behavior.


> **Checkpoint:**
> Your transformer can be used inside a `Pipeline` without errors.

> **Self-check:**
> If `transform` runs twice, do you get stable and predictable output?


In [40]:
# Work on a copy so the original raw dataset stays unchanged for later examples.
seattle_weather_data_sklearn = seattle_weather_data.copy()

In [41]:
from sklearn.base import BaseEstimator, TransformerMixin


class TempMinTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # This transformer does not learn statistics from the data,
        # but scikit-learn still expects every estimator to have a `fit` method.
        self.is_fitted_ = True
        return self

    # Copy first, then convert `temp_min` from Fahrenheit strings to Celsius floats.
    def transform(self, X, y=None):
        # Copying avoids mutating the input DataFrame in place.
        X_transformed = X.copy()

        # For each value we:
        # 1. cast to string,
        # 2. remove the trailing ` F`,
        # 3. convert to float,
        # 4. convert Fahrenheit to Celsius.
        X_transformed["temp_min"] = X_transformed["temp_min"].apply(
            lambda x: (float(str(x).strip(" F")) - 32) / 1.8
        )
        return X_transformed

In [42]:
# Create the transformer object.
temp_min_transformer = TempMinTransformer()

# `fit_transform` is a convenience method that runs `fit` and then `transform`
# on the same training-style dataset.
seattle_weather_data_sklearn = temp_min_transformer.fit_transform(
    seattle_weather_data_sklearn
)

In [43]:
seattle_weather_data_sklearn.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012/01/01,0.0,12.8,5.0,4.710858,drizzle
1,2012/01/02,10.9,NaN,2.8,4.527453,r
2,2012/01/03,0.8,11.7,7.2,2.315752,r.
3,2012/01/03,0.8,11.7,7.2,2.315752,r.
4,2012/01/04,20.3,12.2,5.6,4.719543,r


After dealing with the `temp_min` column we can continue with the `weather` column.


In [44]:
# Inspect the unique raw category values before writing the next cleaning rule.
# This helps us see which abbreviations and spelling variants exist.
seattle_weather_data_sklearn["weather"].unique()

<StringArray>
['drizzle',       'r',      'r.',    'rain',       's',    'Rain',     'sun',
       nan,    'Snow',     'sn.',      'sw',       'd',      's.',     'Sun',
   'driz.',    'snow', 'Drizzle',     'Fog',       'f',      'f.',     'fog']
Length: 21, dtype: str

Also for this step we can write a **custom transformer** in scikit-learn:


In [45]:
class WeatherColumnTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Again, nothing is learned here, but we keep the estimator API consistent.
        self.is_fitted_ = True
        return self

    def transform(self, X, y=None):
        # Copy the full DataFrame and also copy the target column we want to clean.
        X_transformed = X.copy()
        X_temp = X_transformed["weather"].copy()

        # Each regex pattern maps a short messy label to one standard category.
        expressions = {
            r"\br\b": "rain",
            r"\bf\b": "fog",
            r"\b(sw|sn)\b": "snow",
            r"\bs\b": "sun",
            r"\b(d|driz)\b": "drizzle",
        }

        # Loop through the column row by row so we can normalize each string.
        for idx, value in X_temp.items():
            # Missing values may appear as floats (`NaN`), so we skip those.
            if not isinstance(value, float):
                # Normalize case and remove trailing periods like `Rain.`.
                X_temp[idx] = X_temp[idx].lower().strip(".")

                # Apply every replacement rule until the messy label becomes standardized.
                for key, replacement in expressions.items():
                    X_temp[idx] = re.sub(key, replacement, X_temp[idx])

        # Put the cleaned column back into the copied DataFrame.
        X_transformed["weather"] = X_temp
        return X_transformed

In [46]:
# Instantiate the custom transformer for the weather column.
weather_transformer = WeatherColumnTransformer()

# Run the full estimator flow on the current dataset copy.
seattle_weather_data_sklearn = weather_transformer.fit_transform(
    seattle_weather_data_sklearn
)

In [47]:
seattle_weather_data_sklearn["weather"].unique()

<StringArray>
['drizzle', 'rain', 'sun', nan, 'snow', 'fog']
Length: 6, dtype: str

Now let's have a look if every column is the type we are expecting:


In [48]:
# `.info()` is a quick schema check: column names, non-null counts, and data types.
# We use it here to confirm whether the earlier cleaning steps changed types as expected.
seattle_weather_data_sklearn.info()

<class 'pandas.DataFrame'>
RangeIndex: 1498 entries, 0 to 1497
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1498 non-null   str    
 1   precipitation  1396 non-null   str    
 2   temp_max       1144 non-null   float64
 3   temp_min       1498 non-null   float64
 4   wind           1498 non-null   float64
 5   weather        1442 non-null   str    
dtypes: float64(3), str(3)
memory usage: 70.3 KB


So far so good, but there are still two columns that need our attention.  

In a real preprocessing pipeline, numeric columns often arrive as messy strings rather than clean floats. This transformer is your chance to practice writing a small, reusable cleanup step that can be applied the same way during training and on new data later.

The important part is not just converting the column successfully. You also want the transformer to be pipeline-safe: it should copy the input first, make one focused change, and return a transformed DataFrame without mutating the original input.

**Exercise goal:** Implement a custom transformer that cleans `precipitation` and converts it to numeric.

@TODO:
1. Implement `FloatColumnTransformer` with `.fit()` and `.transform()`.
2. Copy the input DataFrame before modifying values.
3. Remove the `$` sign from `precipitation`.
4. Convert `precipitation` to numeric with `pd.to_numeric(..., errors="coerce")`.
5. Return the transformed DataFrame.


In [49]:
# Exercise Starter (do this first)
# TODO: Implement FloatColumnTransformer on your own before checking the solution below.
# Focus on three steps: copy the input, clean the string values, then convert to numeric.
# Hint: Remove `$` and convert with `pd.to_numeric(..., errors="coerce")`.


### Reference solution ([src/pipeline/custom_transformer_seattle_solution.py](src/pipeline/custom_transformer_seattle_solution.py))


In [50]:
class FloatColumnTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No learned parameters, but we keep the standard scikit-learn interface.
        self.is_fitted_ = True
        return self

    def transform(self, X, y=None):
        # Always copy first so we do not unexpectedly modify outside variables.
        X = X.copy()

        # Clean the precipitation column step by step:
        # 1. force values to strings,
        # 2. remove dollar signs if they appear,
        # 3. trim surrounding whitespace,
        # 4. convert to numeric.
        # Invalid values become `NaN` because of `errors="coerce"`.
        X["precipitation"] = pd.to_numeric(
            X["precipitation"]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.strip(),
            errors="coerce",
        )
        return X

Dates are often stored as plain strings in raw datasets, but downstream preprocessing usually becomes much easier once they are converted to a real datetime type. This exercise mirrors a common first cleanup step in production code.

Just like in the previous exercise, keep the transformer small and predictable. The main learning goal is to practice the scikit-learn transformer pattern, not to build a complex date-feature engineering step yet.

**Exercise goal:** Implement a custom transformer that converts `date` to pandas datetime format.

@TODO:
1. Implement `DateColumnTransformer` with `.fit()` and `.transform()`.
2. Copy the input DataFrame before modifying values.
3. Convert `date` with `pd.to_datetime(..., errors="coerce")`.
4. Return the transformed DataFrame.


In [51]:
# Exercise Starter (do this first)
# TODO: Implement DateColumnTransformer on your own before checking the solution below.
# Keep the same pattern as the previous transformer: copy first, then convert one column.
# Hint: Use pd.to_datetime(..., errors="coerce").


### Reference solution ([src/pipeline/custom_transformer_seattle_solution.py](src/pipeline/custom_transformer_seattle_solution.py))


In [52]:
class DateColumnTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # This transformer also follows the estimator API even though it does not learn from data.
        self.is_fitted_ = True
        return self

    def transform(self, X, y=None):
        # Work on a copy to keep the input stable.
        X = X.copy()

        # Convert strings into pandas datetimes.
        # Invalid dates become `NaT` (“Not a Time”) because of `errors="coerce"`.
        # We cast back to `object` so the output stays notebook-friendly for later mixed-type processing.
        X["date"] = pd.to_datetime(X["date"], errors="coerce").astype("object")
        return X

After this, we can **combine all of our custom transformers into a pipeline**.  

Individual transformers are useful, but the real value of scikit-learn comes from chaining them into a repeatable workflow. This step turns several small cleanup operations into one reusable pipeline that can be fitted once and applied consistently later.

The step order matters because each transformer prepares the data for later steps. Treat this exercise as practice in reading a preprocessing workflow from top to bottom and expressing it clearly in pipeline form.

**Exercise goal:** Build a `data_cleaning_pipeline` with the four custom transformer steps in order.

@TODO:
1. Create a pipeline named `data_cleaning_pipeline`.
2. Add steps in this order: `temp_min` -> `weather_strings` -> `precipitation_float` -> `date_datetime`.
3. Use the custom transformer classes defined above.


In [53]:
# Exercise Starter (do this first)
# TODO: Create a Pipeline named data_cleaning_pipeline with these steps:
# temp_min -> weather_strings -> precipitation_float -> date_datetime
# Think of this as writing down the cleanup workflow in executable order.


### Reference solution ([src/pipeline/preprocessing_seattle_weather.py](src/pipeline/preprocessing_seattle_weather.py))


In [54]:
from sklearn.pipeline import Pipeline

# A Pipeline chains multiple transformers in the exact order listed below.
# The output of one step becomes the input to the next step.
data_cleaning_pipeline = Pipeline(
    [
        ("temp_min", TempMinTransformer()),
        ("weather_strings", WeatherColumnTransformer()),
        ("precipitation_float", FloatColumnTransformer()),
        ("date_datetime", DateColumnTransformer()),
    ]
)

Now let's use this pipeline on our data.  

First, we make a new copy and split our data to simulate new incoming data:  


In [55]:
# Start from a fresh copy before demonstrating the full pipeline workflow.
seattle_weather_data_pipeline = seattle_weather_data.copy()

# Split once into a training subset and a stand-in for unseen future data.
# `random_state=42` makes the split reproducible for everyone.
train_weather_data, new_weather_data = train_test_split(
    seattle_weather_data_pipeline, test_size=0.2, random_state=42
)

We will use our pipeline to **fit and transform our training dataset**.  

(Actually, there is not much to fit; in the next section this step will be more important.)  


In [56]:
# Fit the pipeline on the training data and immediately transform that same training data.
# This is the only split the pipeline is allowed to learn from.
train_weather_data = data_cleaning_pipeline.fit_transform(train_weather_data)

Now we can use the already fitted pipeline to **transform our new dataset**.  


In [57]:
# Apply the already-fitted pipeline to new data.
# We call only `transform` here so we reuse the same learned behavior and avoid leakage.
new_weather_data = data_cleaning_pipeline.transform(new_weather_data)

scikit-learn pipelines enable us to conveniently fit and transform our training data and easily apply the same transformation to new data.  

One of the main advantages of this process will become clearer in the next sections.  


## Dealing with missing data

Now let's see how we use a transformer to deal with missing values in our data.  

We will continue to work with our `train_weather_data` and `new_weather_data` splits.  


### Visual Guide: Fit/Transform Separation

```mermaid
flowchart TD
    A["Training split"]
    B["Fit imputers and encoders"]
    C["Transform training split"]
    D["New or validation split"]
    E["Reuse fitted transformers"]
    F["Leakage-safe transformed outputs"]

    A --> B --> C --> F
    D --> E --> F
    B --> E
```


In [58]:
# Display number of missing values per column
train_weather_data.isna().sum()

date               0
precipitation     80
temp_max         285
temp_min           0
wind               0
weather           40
dtype: int64

In [59]:
# Find the percentage of missing data per variable
train_weather_data.isnull().mean() * 100

date              0.000000
precipitation     6.677796
temp_max         23.789649
temp_min          0.000000
wind              0.000000
weather           3.338898
dtype: float64

### Imputing data


#### Mean/Median/Mode imputation

We can use the `SimpleImputer` from scikit-learn to impute our missing values.  


In [60]:
# Make copies so the imputation example does not overwrite the previous cleaned datasets.
train_weather_data_sklearn = train_weather_data.copy()
new_weather_data_sklearn = new_weather_data.copy()

In [61]:
# Import `SimpleImputer` to fill in missing values with a consistent sklearn API.
from sklearn.impute import SimpleImputer

# These are the columns where median imputation makes sense because they are numeric.
numerical_features = ["precipitation", "temp_max", "temp_min", "wind"]

# Create the imputer object.
# `strategy="median"` means one median will be learned for each numeric column.
num_imputer = SimpleImputer(strategy="median")

# Fit only on the training set.
# During fitting, the imputer stores the median of each listed column.
num_imputer.fit(train_weather_data_sklearn[numerical_features])

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [62]:
# Replace missing numeric values in both splits using the medians learned from train only.
# `transform` returns a NumPy array, so we assign that array back into the same DataFrame columns.
train_weather_data_sklearn[numerical_features] = num_imputer.transform(
    train_weather_data_sklearn[numerical_features]
)
new_weather_data_sklearn[numerical_features] = num_imputer.transform(
    new_weather_data_sklearn[numerical_features]
)

> **Note:** After transforming the data with the `SimpleImputer`, it is returned as a **NumPy array**.  

We imputed all the numerical features, now let's impute the categorical ones.


In [63]:
# The `weather` column is categorical, so we treat it separately.
categorical_features = ["weather"]

# `most_frequent` is the categorical equivalent of filling with the mode.
cat_imputer = SimpleImputer(strategy="most_frequent")

# Fit on train only so the replacement value comes from the training distribution.
cat_imputer.fit(train_weather_data_sklearn[categorical_features])

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [64]:
# Use the fitted categorical imputer on both datasets.
# The missing `weather` labels are replaced with the most common training category.
train_weather_data_sklearn[categorical_features] = cat_imputer.transform(
    train_weather_data_sklearn[categorical_features]
)
new_weather_data_sklearn[categorical_features] = cat_imputer.transform(
    new_weather_data_sklearn[categorical_features]
)

In [65]:
# Display number of missing values per column in the train data.
train_weather_data_sklearn.isna().sum()

date             0
precipitation    0
temp_max         0
temp_min         0
wind             0
weather          0
dtype: int64

In [66]:
# Display number of missing values per column in the new data.
new_weather_data_sklearn.isna().sum()

date             0
precipitation    0
temp_max         0
temp_min         0
wind             0
weather          0
dtype: int64

## @TODO Practice 1 (scikit-learn): Reusable Imputer Workflow

In notebook 01, you computed imputation values manually with pandas. Here, the goal is to practice the scikit-learn version of the same idea: fit the imputer once on training data, then reuse it on both training and new data without recalculating statistics.

The key concept is not the imputation strategy itself, but the workflow. Fitting on all available data causes leakage. This exercise is about building the habit of fitting on train data only and reusing the fitted transformer afterward.

**Exercise goal:** Practice train-only fitting and consistent transforms for numerical + categorical imputations.

@TODO:
1. Create train/new copies.
2. Build and fit a median `SimpleImputer` for numerical features.
3. Build and fit a mode `SimpleImputer` for categorical features.
4. Transform both train and new data with the fitted imputers.

**Hints:**
- Fit imputers on training data only.
- Reuse fitted imputers to transform new data.
- Track numerical and categorical feature lists explicitly.

> **Checkpoint:**
> Missing-value handling is consistent between train and new datasets.


In [67]:
# Exercise Starter (do this first)
# TODO: Implement a reusable imputer workflow with scikit-learn.
# The important habit here is: fit on train data once, then reuse the fitted imputers.

practice_train_sklearn = train_weather_data.copy()
practice_new_sklearn = new_weather_data.copy()

# Keep the numeric and categorical columns separate because they need different strategies.
numerical_features_practice = ["precipitation", "temp_max", "temp_min", "wind"]
categorical_features_practice = ["weather"]

# @TODO 1-3: Define the imputers and fit them on training data only.
# After fitting, each imputer should have learned one replacement value per column.
num_imputer_practice = None
cat_imputer_practice = None

# @TODO 4: Transform both train and new splits with the fitted imputers.
# Assign the transformed arrays back into the same feature columns.
# The final line below is a quick check that your missing values are gone.

practice_train_sklearn.isna().sum(), practice_new_sklearn.isna().sum()

(date               0
 precipitation     80
 temp_max         285
 temp_min           0
 wind               0
 weather           40
 dtype: int64,
 date              0
 precipitation    23
 temp_max         69
 temp_min          0
 wind              0
 weather          16
 dtype: int64)

### Reference solution  


In [68]:
# Reference solution
practice_train_sklearn_solution = train_weather_data.copy()
practice_new_sklearn_solution = new_weather_data.copy()

numerical_features_practice_solution = ["precipitation", "temp_max", "temp_min", "wind"]
categorical_features_practice_solution = ["weather"]

# Create one imputer for numeric columns and one for categorical columns.
num_imputer_practice_solution = SimpleImputer(strategy="median")
cat_imputer_practice_solution = SimpleImputer(strategy="most_frequent")

# Fit both imputers on training data only.
num_imputer_practice_solution.fit(
    practice_train_sklearn_solution[numerical_features_practice_solution]
)
cat_imputer_practice_solution.fit(
    practice_train_sklearn_solution[categorical_features_practice_solution]
)

# Transform the numeric columns in both splits with the medians learned from train.
practice_train_sklearn_solution[numerical_features_practice_solution] = (
    num_imputer_practice_solution.transform(
        practice_train_sklearn_solution[numerical_features_practice_solution]
    )
)
practice_new_sklearn_solution[numerical_features_practice_solution] = (
    num_imputer_practice_solution.transform(
        practice_new_sklearn_solution[numerical_features_practice_solution]
    )
)

# Transform the categorical columns in both splits with the most frequent training value.
practice_train_sklearn_solution[categorical_features_practice_solution] = (
    cat_imputer_practice_solution.transform(
        practice_train_sklearn_solution[categorical_features_practice_solution]
    )
)
practice_new_sklearn_solution[categorical_features_practice_solution] = (
    cat_imputer_practice_solution.transform(
        practice_new_sklearn_solution[categorical_features_practice_solution]
    )
)

# Both outputs should now show zero missing values for the imputed columns.
practice_train_sklearn_solution.isna().sum(), practice_new_sklearn_solution.isna().sum()

(date             0
 precipitation    0
 temp_max         0
 temp_min         0
 wind             0
 weather          0
 dtype: int64,
 date             0
 precipitation    0
 temp_max         0
 temp_min         0
 wind             0
 weather          0
 dtype: int64)

During the fitting step, the imputer will learn and save the median/mode for each feature separately, so we don't have to take care about calculating it repeatedly or hardcode it somewhere.  

Sometimes it still makes sense to create your own transformer that imputes missing values with a more sophisticated strategy. In our case we could, for example, calculate the median for each month separately and impute the missing values for each month with the corresponding value.  

Of course, we can add the imputation steps also to our pipeline. Therefore, we need to create **one pipeline for numerical features and one for categorical features**. The steps of our pipeline are defined in a list. Each step is composed of a tuple with a name (that we can choose) and the transformer.  


In [69]:
# Pipeline for numerical features.
# Right now this pipeline has one step, but wrapping it in `Pipeline`
# makes it easy to add more numeric preprocessing later.
num_pipeline = Pipeline(
    [
        ("imputer_num", SimpleImputer(strategy="median")),
    ]
)

# Pipeline for categorical features.
cat_pipeline = Pipeline(
    [
        ("imputer_cat", SimpleImputer(strategy="most_frequent")),
    ]
)

In the end, **both pipelines are combined into one** pipeline called "preprocessor" using `ColumnTransformer` from scikit-learn.  

The `ColumnTransformer` applies transformers (num_pipeline / cat_pipeline) to specific columns of an array or DataFrame (num_features / cat_features).  

We have to add the `remainder="passthrough"` to not drop the `date` column but rather leave it untouched.  


In [70]:
from sklearn.compose import ColumnTransformer

# `ColumnTransformer` lets us send different groups of columns
# through different preprocessing pipelines in one combined object.
imputing_missing_pipe = ColumnTransformer(
    [
        ("num", num_pipeline, numerical_features),
        ("cat", cat_pipeline, categorical_features),
    ],
    # Keep any columns not listed above, such as `date`, unchanged.
    remainder="passthrough",
)

Now we can also **combine our preprocessor with our data cleaning pipeline**.  


In [71]:
# This outer pipeline first runs the custom cleaning steps,
# then runs the missing-value preprocessing on the cleaned output.
preprocessor_pipe = Pipeline(
    [
        ("data_cleaning", data_cleaning_pipeline),
        ("missing_values", imputing_missing_pipe),
    ]
)

In [72]:
preprocessor_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('data_cleaning', ...), ('missing_values', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('temp_min', ...), ('weather_strings', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer

Let's see the whole pipeline in action on a fresh copy of the data (keep in mind the output will be a NumPy array):  


In [73]:
# Start again from raw data so we can demonstrate the full combined pipeline end to end.
seattle_weather_data_pipeline = seattle_weather_data.copy()

# Recreate the train/new split for this independent example.
train_weather_data, new_weather_data = train_test_split(
    seattle_weather_data_pipeline, test_size=0.2, random_state=42
)

In [74]:
# Fit the full preprocessing pipeline on train.
train_weather_data = preprocessor_pipe.fit_transform(train_weather_data)

# Reuse that fitted pipeline on the new split without refitting.
new_weather_data = preprocessor_pipe.transform(new_weather_data)

If we now validate the DataFrame with our Pydantic model from the top of the notebook, we don't get errors.  

But before we can use the model, we need to **convert the transformed NumPy array back to a pandas DataFrame**.  


In [75]:
# The pipeline output is currently a NumPy array, so we rebuild DataFrames for readability.
train_weather_data = pd.DataFrame(train_weather_data)
new_weather_data = pd.DataFrame(new_weather_data)

# `ColumnTransformer` prefixes feature names with the transformer name
# (for example `num__temp_max`). We strip those prefixes to recover cleaner labels.
column_lst = [
    x.replace("num__", "").replace("cat__", "").replace("remainder__", "")
    for x in imputing_missing_pipe.get_feature_names_out().tolist()
]

train_weather_data.columns = column_lst
new_weather_data.columns = column_lst
train_weather_data

,precipitation,temp_max,temp_min,wind,weather,date
0,26.2,15.6,8.3,4.625898,fog,2015-02-05
1,0.0,21.7,11.7,2.133832,sun,2015-05-26
2,3.0,15.6,7.2,4.312184,sun,2014-12-19
3,0.5,8.9,3.9,3.811336,rain,2013-02-26
4,0.0,15.6,-0.5,0.92283,sun,2014-11-18
...,...,...,...,...,...,...
1193,0.0,7.8,5.6,1.605368,fog,2015-01-07
1194,0.0,22.8,11.1,3.0126,sun,2015-06-16
1195,13.7,11.7,5.6,4.736475,fog,2014-04-19
1196,7.1,6.7,2.8,4.507815,fog,2015-11-24


In [76]:
data_validation(train_weather_data, DataValidation)

,precipitation,temp_max,temp_min,wind,weather,date
0,26.2,15.6,8.3,4.625898,fog,2015-02-05
1,0.0,21.7,11.7,2.133832,sun,2015-05-26
2,3.0,15.6,7.2,4.312184,sun,2014-12-19
3,0.5,8.9,3.9,3.811336,rain,2013-02-26
4,0.0,15.6,-0.5,0.92283,sun,2014-11-18
...,...,...,...,...,...,...
1193,0.0,7.8,5.6,1.605368,fog,2015-01-07
1194,0.0,22.8,11.1,3.0126,sun,2015-06-16
1195,13.7,11.7,5.6,4.736475,fog,2014-04-19
1196,7.1,6.7,2.8,4.507815,fog,2015-11-24


## Further preprocessing steps


### Feature transformation of categorical features

One-hot encoding has its own transformer in scikit-learn, so we will use that one instead of the `get_dummies` function from notebook 01.  


In [77]:
from sklearn.preprocessing import OneHotEncoder

# Create the encoder.
# `drop="first"` removes one dummy column to avoid redundant information.
# `handle_unknown="ignore"` keeps the transform step safe if new data contains unseen categories.
one_hot_encoder = OneHotEncoder(
    drop="first",  # To return k-1; use drop=False to return k dummies
    handle_unknown="ignore",  # If we encounter an unknown category in new data the category will be ignored and set to 0
    sparse_output=False,
)

# Fit on train only so the encoder learns the category vocabulary from training data.
one_hot_encoder.fit(train_weather_data[categorical_features])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_c

In [78]:
# Transform both datasets into dummy-variable matrices using the learned categories.
train_weather_data_enc = one_hot_encoder.transform(
    train_weather_data[categorical_features]
)

new_weather_data_enc = one_hot_encoder.transform(new_weather_data[categorical_features])

In [79]:
# Convert the NumPy arrays back to DataFrames so the encoded columns stay easy to inspect.
train_weather_data_enc = pd.DataFrame(train_weather_data_enc)
new_weather_data_enc = pd.DataFrame(new_weather_data_enc)

# Recover readable column names such as `weather_rain` from the fitted encoder.
train_weather_data_enc.columns = one_hot_encoder.get_feature_names_out(
    ["weather"]
).tolist()
new_weather_data_enc.columns = one_hot_encoder.get_feature_names_out(
    ["weather"]
).tolist()
train_weather_data_enc

,weather_fog,weather_rain,weather_snow,weather_sun
0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0
...,...,...,...,...
1193,1.0,0.0,0.0,0.0
1194,0.0,0.0,0.0,1.0
1195,1.0,0.0,0.0,0.0
1196,1.0,0.0,0.0,0.0


## @TODO Practice 2 (scikit-learn): One-Hot Encoding With Feature Names

One-hot encoding is easy to apply, but beginners often lose track of the resulting feature names or accidentally build train and new outputs that are hard to compare. This exercise adds the important missing step of turning the encoded arrays back into readable DataFrames.

The main skill here is not only using `OneHotEncoder`, but rebuilding the result in a way that stays interpretable. Make sure the feature names come from the fitted encoder and that the DataFrame indices still line up with the original rows.

**Exercise goal:** Encode categorical columns with scikit-learn and keep interpretable feature names.

@TODO:
1. Fit `OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)` on train data.
2. Transform train and new data.
3. Convert arrays to DataFrames using `get_feature_names_out(...)`.

**Hints:**
- Call `.fit(...)` only on training data.
- Reuse the same encoder for new data.
- Preserve index alignment when rebuilding DataFrames.

> **Checkpoint:**
> Encoded train/new outputs share a compatible set of one-hot columns.



In [80]:
# Exercise Starter (do this first)
# TODO: One-hot encode `weather` and preserve feature names.
# The main goal is to keep the encoded output interpretable and aligned to the original rows.

categorical_features_practice = ["weather"]

# @TODO 1: Define and fit the encoder on training data only.
one_hot_encoder_practice = None

# @TODO 2: Transform the train and new sets with the fitted encoder.
train_weather_data_enc_practice = None
new_weather_data_enc_practice = None

# @TODO 3: Build DataFrames with feature names and matching indices.
train_weather_data_enc_practice_df = None
new_weather_data_enc_practice_df = None

train_weather_data_enc_practice_df, new_weather_data_enc_practice_df

(None, None)

### Reference solution


In [81]:
# Reference solution
categorical_features_practice_solution = ["weather"]

one_hot_encoder_practice_solution = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False,
)
one_hot_encoder_practice_solution.fit(
    train_weather_data[categorical_features_practice_solution]
)

train_weather_data_enc_practice_solution = one_hot_encoder_practice_solution.transform(
    train_weather_data[categorical_features_practice_solution]
)
new_weather_data_enc_practice_solution = one_hot_encoder_practice_solution.transform(
    new_weather_data[categorical_features_practice_solution]
)

feature_names_practice_solution = (
    one_hot_encoder_practice_solution.get_feature_names_out(
        categorical_features_practice_solution
    )
)

train_weather_data_enc_practice_df_solution = pd.DataFrame(
    train_weather_data_enc_practice_solution,
    columns=feature_names_practice_solution,
    index=train_weather_data.index,
)
new_weather_data_enc_practice_df_solution = pd.DataFrame(
    new_weather_data_enc_practice_solution,
    columns=feature_names_practice_solution,
    index=new_weather_data.index,
)

(
    train_weather_data_enc_practice_df_solution.head(),
    new_weather_data_enc_practice_df_solution.head(),
)

(   weather_fog  weather_rain  weather_snow  weather_sun
 0          1.0           0.0           0.0          0.0
 1          0.0           0.0           0.0          1.0
 2          0.0           0.0           0.0          1.0
 3          0.0           1.0           0.0          0.0
 4          0.0           0.0           0.0          1.0,
    weather_fog  weather_rain  weather_snow  weather_sun
 0          0.0           0.0           0.0          1.0
 1          0.0           0.0           0.0          1.0
 2          1.0           0.0           0.0          0.0
 3          1.0           0.0           0.0          0.0
 4          0.0           0.0           0.0          1.0)

We get the same output as before, but using the scikit-learn implementation has the advantage that **unknown categories in new data can be handled without raising an error** or needing us to intervene.  


### Feature transformation of numerical features

Since standardization is a commonly used technique to scale your data, scikit-learn offers their own implementation, the `StandardScaler`.  

In order to apply square root transformation to the `wind` column, we have to use the `FunctionTransformer`.  

We will start again with the **square root transformation**.  


In [82]:
numerical_features

['precipitation', 'temp_max', 'temp_min', 'wind']

In [83]:
from sklearn.preprocessing import FunctionTransformer

# `FunctionTransformer` wraps a normal Python/NumPy function so it behaves
# like a scikit-learn transformer and can live inside a pipeline.
sqrt_transformer = FunctionTransformer(np.sqrt, validate=True)

In [84]:
# Learn any needed validation details on the training slice, then apply the same square-root step.
# Here the transformation itself is deterministic, but we still keep the train/new workflow consistent.
train_weather_sqrt = sqrt_transformer.fit_transform(train_weather_data[["wind"]])
new_weather_data_sqrt = sqrt_transformer.transform(new_weather_data[["wind"]])

In [85]:
# Convert the transformed arrays back to DataFrames so the single feature keeps its name.
train_weather_sqrt = pd.DataFrame(train_weather_sqrt)
new_weather_data_sqrt = pd.DataFrame(new_weather_data_sqrt)

train_weather_sqrt.columns = ["wind"]
new_weather_data_sqrt.columns = ["wind"]
train_weather_sqrt

,wind
0,2.150790
1,1.460764
2,2.076580
3,1.952264
4,0.960641
...,...
1193,1.267031
1194,1.735684
1195,2.176344
1196,2.123162


Followed by the **standardization**:


In [86]:
from sklearn.preprocessing import StandardScaler

# `StandardScaler` rescales each feature to mean 0 and standard deviation 1.
standard_scaler = StandardScaler()

# Fit only on training data so the scaling statistics come from train.
train_weather_scaled = standard_scaler.fit_transform(
    train_weather_data[numerical_features]
)

# Apply the same learned scaling to new data.
new_weather_data_scaled = standard_scaler.transform(
    new_weather_data[numerical_features]
)

In [87]:
# Rebuild DataFrames for readability after scaling returns NumPy arrays.
train_weather_scaled = pd.DataFrame(train_weather_scaled)
new_weather_data_scaled = pd.DataFrame(new_weather_data_scaled)

# `get_feature_names_out()` gives back the feature names in the same output order.
train_weather_scaled.columns = standard_scaler.get_feature_names_out().tolist()
new_weather_data_scaled.columns = standard_scaler.get_feature_names_out().tolist()
train_weather_scaled

,precipitation,temp_max,temp_min,wind
0,3.743847,-0.123130,-0.001396,0.982994
1,-0.441129,0.808160,0.675421,-0.785331
2,0.038066,-0.123130,-0.220366,0.760389
3,-0.361263,-1.146023,-0.877277,0.404996
4,-0.441129,-0.123130,-1.753158,-1.644636
...,...,...,...,...
1193,-0.441129,-1.313961,-0.538868,-1.160319
1194,-0.441129,0.976098,0.555983,-0.161773
1195,1.747198,-0.718546,-0.538868,1.061458
1196,0.692967,-1.481899,-1.096247,0.899205


Of course, we can extend our pipeline with these steps as well.  

The one-hot encoder can be added to the `cat_pipeline` from above.  

The pipeline step for numerical features needs to be split into two parts now: one for features with square root transformation and one for those without.  


In [88]:
# Pipeline for numeric features that only need imputation + scaling.
num_impute_scaling_pipeline = Pipeline(
    [
        ("imputer_num", SimpleImputer(strategy="median")),
        ("std_scaling", StandardScaler()),
    ]
)

# Separate numeric pipeline for the `wind` column,
# because that feature gets an extra square-root transformation before scaling.
num_impute_sqrt_scaling_pipeline = Pipeline(
    [
        ("imputer_num", SimpleImputer(strategy="median")),
        ("sqrt", FunctionTransformer(np.sqrt, validate=True)),
        ("std_scaling", StandardScaler()),
    ]
)

# Categorical pipeline: fill missing values, then one-hot encode categories.
cat_pipeline = Pipeline(
    [
        ("imputer_cat", SimpleImputer(strategy="most_frequent")),
        (
            "one_hot",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

In [89]:
# Combine all feature-specific pipelines into one column-wise preprocessor.
preprocessor_pipe = ColumnTransformer(
    [
        # All numeric columns except `wind` use the standard numeric branch.
        ("num_scaling_impute", num_impute_scaling_pipeline, numerical_features[:-1]),
        (
            # The last numeric feature (`wind`) uses the sqrt-specific branch.
            "num_sqrt_scaling_impute",
            num_impute_sqrt_scaling_pipeline,
            numerical_features[-1:],
        ),
        # Categorical weather values go through imputation and one-hot encoding.
        ("cat", cat_pipeline, categorical_features),
    ],
    # Pass through any remaining columns, such as `date`.
    remainder="passthrough",
)

In [90]:
# Final full pipeline:
# 1. clean raw columns,
# 2. run column-specific preprocessing,
# 3. return a model-ready matrix.
preprocessor_pipe = Pipeline(
    [("data_cleaning", data_cleaning_pipeline), ("preprocessor", preprocessor_pipe)]
)

Let's have a look at our pipeline now:  


In [91]:
preprocessor_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('data_cleaning', ...), ('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('temp_min', ...), ('weather_strings', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer :

And finally **transform the data**:


In [92]:
# Run the full preprocessing workflow on the complete dataset.
# In a real ML project, you would usually fit on train only, but here we use all rows
# because this final cell is just demonstrating the pipeline output structure.
seattle_weather_data_clean = preprocessor_pipe.fit_transform(seattle_weather_data)

# Rebuild a labeled DataFrame so the final pipeline output stays easy to inspect.
# We assemble the output names manually because the sqrt branch uses a
# `FunctionTransformer`, which does not expose clean output names here.
cat_feature_names = (
    preprocessor_pipe.named_steps["preprocessor"]
    .named_transformers_["cat"]
    .named_steps["one_hot"]
    .get_feature_names_out(categorical_features)
    .tolist()
)
remaining_features = [
    col
    for col in seattle_weather_data.columns
    if col not in numerical_features + categorical_features
]
feature_names = (
    numerical_features[:-1]
    + numerical_features[-1:]
    + cat_feature_names
    + remaining_features
)
seattle_weather_data_clean = pd.DataFrame(
    seattle_weather_data_clean,
    columns=feature_names,
    index=seattle_weather_data.index,
)

In [93]:
seattle_weather_data_clean.head()

,precipitation,temp_max,temp_min,wind,weather_fog,weather_rain,weather_snow,weather_sun,date
0,-0.435593,-0.53148,-0.648232,1.045955,0.0,0.0,0.0,0.0,2012-01-01
1,1.307568,-0.099892,-1.087785,0.937067,0.0,1.0,0.0,0.0,2012-01-02
2,-0.307654,-0.701033,-0.20868,-0.609418,0.0,1.0,0.0,0.0,2012-01-03
3,-0.307654,-0.701033,-0.20868,-0.609418,0.0,1.0,0.0,0.0,2012-01-03
4,2.810845,-0.623964,-0.528354,1.051058,0.0,1.0,0.0,0.0,2012-01-04


## Next steps

The next step is to refactor this notebook workflow into reusable Python files.

- Exercise starter file: `src/pipeline/custom_transformer_seattle.py`
- Reference solution file: `src/pipeline/custom_transformer_seattle_solution.py`
- Pipeline class example: `src/pipeline/preprocessing_seattle_weather.py`

> **Self-check:**
> You should be able to explain where each transformation belongs: custom transformer, column pipeline, or full preprocessing pipeline.

If you want to go deeper into feature engineering, the [Python Feature Engineering Cookbook](https://www.packtpub.com/product/python-feature-engineering-cookbook/9781789806311) and its [companion GitHub repository](https://github.com/PacktPublishing/Python-Feature-Engineering-Cookbook/tree/master) provide additional examples.


### **Quiz**

1. **What is one main advantage of using scikit-learn pipelines over manual feature engineering in pandas?**  
    [ ] Pipelines are always faster than pandas  
    [ ] Pipelines allow modular, reusable, and consistent transformations  
    [ ] Pipelines can only be used with tree-based models  
    [ ] Pipelines remove the need for model evaluation  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Pipelines allow modular, reusable, and consistent transformations  
      
      **Description:** Pipelines structure preprocessing into reusable steps, ensuring the same transformations are applied during training and inference.
    </details>

---

2. **Which of the following is NOT a benefit of scikit-learn pipelines?**  
    [ ] Integration with scikit-learn estimators  
    [ ] Ability to include custom transformers  
    [ ] Automatic feature importance calculation  
    [ ] Efficiency and parallelization  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Automatic feature importance calculation  
      
      **Description:** Pipelines handle transformations and model integration but do not calculate feature importance automatically — that depends on the chosen model.
    </details>

---

3. **Which code snippet correctly creates and fits a simple pipeline with a `StandardScaler` followed by a `LinearRegression` model?**  
    [ ]  
    ```python
    pipe = Pipeline([
        ('scaler', StandardScaler),
        ('model', LinearRegression)
    ])
    pipe.fit(X, y)
    ```  
    [ ]  
    ```python
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ])
    pipe.fit(X, y)
    ```  
    [ ]  
    ```python
    pipe = Pipeline({
        'scaler': StandardScaler(),
        'model': LinearRegression()
    })
    pipe.train(X, y)
    ```  
    [ ]  
    ```python
    pipe = LinearRegression(StandardScaler())
    pipe.fit(X, y)
    ```  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:**  
      ```python
      pipe = Pipeline([
          ('scaler', StandardScaler()),
          ('model', LinearRegression())
      ])
      pipe.fit(X, y)
      ```  
      
      **Description:** Steps in a scikit-learn pipeline must be ordered as tuples `(name, transformer/estimator)`. Both `StandardScaler` and `LinearRegression` must be instantiated.
    </details>

---

4. **When using pipelines, why is it important to avoid applying transformations directly to the test set before fitting?**  
    [ ] It makes the code longer  
    [ ] It risks data leakage by letting test data influence training transformations  
    [ ] It increases computation time  
    [ ] It prevents models from converging  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** It risks data leakage by letting test data influence training transformations  
      
      **Description:** The pipeline ensures transformations (e.g., scaling) are fitted only on the training set and applied consistently to unseen data, preventing leakage.
    </details>

---

5. **Which of the following best demonstrates creating a custom transformer for scikit-learn pipelines?**  
    [ ]  
    ```python
    class MyTransformer:
        def transform(self, X):
            return X + 1
    ```  
    [ ]  
    ```python
    class MyTransformer(BaseEstimator, TransformerMixin):
        def fit(self, X, y=None):
            return self
        def transform(self, X, y=None):
            return X + 1
    ```  
    [ ]  
    ```python
    def my_transformer(X):
        return X + 1
    ```  
    [ ]  
    ```python
    class MyTransformer(Pipeline):
        def transform(self, X):
            return X + 1
    ```  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:**  
      ```python
      class MyTransformer(BaseEstimator, TransformerMixin):
          def fit(self, X, y=None):
              return self
          def transform(self, X, y=None):
              return X + 1
      ```  
      
      **Description:** Custom transformers must inherit from `BaseEstimator` and `TransformerMixin` to integrate seamlessly with scikit-learn pipelines.
    </details>

---

6. **What happens when you call `pipe.fit(X_train, y_train)` on a pipeline with multiple steps?**  
    [ ] Only the final model is trained  
    [ ] Each transformer in the pipeline is fitted sequentially, then the model is trained on the transformed data  
    [ ] All steps are fitted in parallel and combined at the end  
    [ ] Transformers are skipped and only the estimator is fitted  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Each transformer in the pipeline is fitted sequentially, then the model is trained on the transformed data  
      
      **Description:** The pipeline chains preprocessing and modeling — each transformation is applied in order, with the final estimator trained on the fully transformed dataset.
    </details>

---

7. **Which pipeline feature is especially helpful when performing hyperparameter tuning with `GridSearchCV` or `RandomizedSearchCV`?**  
    [ ] Pipelines can automatically choose the best estimator  
    [ ] Pipelines allow hyperparameters of preprocessing steps and models to be tuned together  
    [ ] Pipelines reduce the number of parameters to tune  
    [ ] Pipelines store training and test sets internally  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Pipelines allow hyperparameters of preprocessing steps and models to be tuned together  
      
      **Description:** With pipelines, both preprocessing parameters (e.g., `StandardScaler`, `PolynomialFeatures`) and model hyperparameters can be searched in a single unified process.
    </details>
